# Assignment 02: Calculus and Optimization

**Module:** 02 — Math for Deep Learning  
**Estimated Time:** 8-10 hours  
**Language:** Python (NumPy for core, PyTorch for verification)

---

## Learning Objectives

- Implement numerical differentiation (forward, central, complex step) and understand floating-point limitations
- Compute backpropagation by hand through a computation graph and verify with PyTorch
- Visualize gradient descent on 2D functions (Rosenbrock, Rastrigin, ill-conditioned quadratic)
- Implement and compare optimizers: Vanilla GD, Momentum, RMSProp, Adam
- Understand convexity, local minima, and saddle points

### Key Equations

**Finite differences:**
$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad \text{(forward, } O(h) \text{)}$$
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad \text{(central, } O(h^2) \text{)}$$

**Chain rule:** $\frac{dL}{dx} = \frac{dL}{dz_3} \cdot \frac{dz_3}{dz_2} \cdot \frac{dz_2}{dz_1} \cdot \frac{dz_1}{dx}$

**Adam update:**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch

sys.path.insert(0, '../../')
from shared_utils.common import set_seed

set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Setup complete.')

---
## Exercise 1: Numerical Differentiation

### 1a: Three Finite Difference Methods

In [ ]:
def forward_difference(f, x: float, h: float = 1e-7) -> float:
    """f'(x) ~ [f(x+h) - f(x)] / h. Accuracy: O(h)."""
    # YOUR CODE HERE
    pass


def central_difference(f, x: float, h: float = 1e-7) -> float:
    """f'(x) ~ [f(x+h) - f(x-h)] / (2h). Accuracy: O(h^2)."""
    # YOUR CODE HERE
    pass


def complex_step(f, x: float, h: float = 1e-20) -> float:
    """f'(x) ~ Im[f(x + ih)] / h. No subtractive cancellation."""
    # YOUR CODE HERE
    # Hint: call f with x + 1j * h, take imaginary part
    pass

In [ ]:
# --- Test 1a ---
f = np.sin
true_deriv = np.cos(1.0)
x = 1.0

print(f'True derivative: {true_deriv:.15f}')
print(f'Forward diff:    {forward_difference(f, x):.15f}')
print(f'Central diff:    {central_difference(f, x):.15f}')
print(f'Complex step:    {complex_step(f, x):.15f}')

assert abs(forward_difference(f, x) - true_deriv) < 1e-5, 'Forward diff failed'
assert abs(central_difference(f, x) - true_deriv) < 1e-9, 'Central diff failed'
assert abs(complex_step(f, x) - true_deriv) < 1e-14, 'Complex step failed'
print('Test 1a passed!')

In [ ]:
# --- 1b: Accuracy comparison (log-log plot) ---
hs = np.logspace(-1, -15, 30)
errors_forward = []
errors_central = []
errors_complex = []

for h in hs:
    errors_forward.append(abs(forward_difference(f, x, h) - true_deriv))
    errors_central.append(abs(central_difference(f, x, h) - true_deriv))
    errors_complex.append(abs(complex_step(f, x, h) - true_deriv))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(hs, errors_forward, 'o-', label='Forward difference', markersize=3)
ax.loglog(hs, errors_central, 's-', label='Central difference', markersize=3)
ax.loglog(hs, errors_complex, '^-', label='Complex step', markersize=3)
ax.set_xlabel('Step size h')
ax.set_ylabel('Absolute error')
ax.set_title('Numerical Differentiation: Accuracy vs Step Size')
ax.legend()
ax.invert_xaxis()
plt.tight_layout()
plt.show()

**Why does forward difference accuracy worsen for very small h? Why doesn't complex step have this problem?**

*YOUR ANSWER HERE (3-5 sentences)*

### 1c: Gradient Checking for DL Activation Functions

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def sigmoid_fn(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid_fn(x)
    return s * (1 - s)

def softplus(x):
    return np.log(1 + np.exp(x))

def softplus_deriv(x):
    return sigmoid_fn(x)

def tanh_fn(x):
    return np.tanh(x)

def tanh_deriv(x):
    return 1 - np.tanh(x) ** 2


def check_gradient(f, f_prime, name, x_range=(-5, 5), n_points=10):
    """Verify analytical gradient against numerical gradient."""
    np.random.seed(42)
    x_test = np.random.uniform(x_range[0], x_range[1], n_points)
    max_error = 0
    for xi in x_test:
        analytical = f_prime(xi)
        numerical = central_difference(f, xi)
        error = abs(analytical - numerical)
        max_error = max(max_error, error)
    print(f'{name}: max error = {max_error:.2e}')
    assert max_error < 1e-5, f'Gradient check FAILED for {name}!'
    return max_error

In [ ]:
# --- Test 1c ---
functions = [
    (relu, relu_deriv, 'ReLU'),
    (sigmoid_fn, sigmoid_deriv, 'Sigmoid'),
    (softplus, softplus_deriv, 'Softplus'),
    (tanh_fn, tanh_deriv, 'Tanh'),
]

# Verify gradients
for f_fn, f_d, name in functions:
    check_gradient(f_fn, f_d, name, x_range=(0.1, 5))

# Plot all functions and derivatives
x_plot = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (f_fn, f_d, name) in zip(axes.flat, functions):
    ax.plot(x_plot, f_fn(x_plot), 'b-', linewidth=2, label=f'{name}(x)')
    ax.plot(x_plot, f_d(x_plot), 'r--', linewidth=2, label=f"{name}'(x)")
    ax.axhline(0, color='gray', alpha=0.3)
    ax.axvline(0, color='gray', alpha=0.3)
    ax.set_title(name)
    ax.legend()
plt.tight_layout()
plt.show()
print('Exercise 1c passed!')

---
## Exercise 2: The Chain Rule — Backpropagation by Hand

### 2a: Simple Computation Graph

$$x = 2.0 \rightarrow z_1 = x^2 \rightarrow z_2 = z_1 + 3 \rightarrow z_3 = \sin(z_2) \rightarrow L = z_3^2$$

**Forward pass (compute by hand first):**
- $z_1 = 4.0$
- $z_2 = 7.0$
- $z_3 = \sin(7.0)$
- $L = \sin^2(7.0)$

**Chain rule:**
$$\frac{dL}{dx} = \frac{dL}{dz_3} \cdot \frac{dz_3}{dz_2} \cdot \frac{dz_2}{dz_1} \cdot \frac{dz_1}{dx} = 2z_3 \cdot \cos(z_2) \cdot 1 \cdot 2x$$

In [ ]:
# --- 2a: Manual computation ---
# YOUR CODE HERE: compute the chain rule result numerically
# Then verify with PyTorch:

x_t = torch.tensor(2.0, requires_grad=True)
z1_t = x_t ** 2
z2_t = z1_t + 3
z3_t = torch.sin(z2_t)
L_t = z3_t ** 2
L_t.backward()

print(f'PyTorch dL/dx = {x_t.grad.item():.6f}')
# print(f'Your dL/dx    = ???')  # Replace with your hand-computed value

### 2b: 3-Layer Neural Network — Full Backpropagation

Architecture:
- Layer 1: $z_1 = w_1 x + b_1$, $a_1 = \sigma(z_1)$
- Layer 2: $z_2 = w_2 a_1 + b_2$, $a_2 = \sigma(z_2)$  
- Layer 3: $z_3 = w_3 a_2 + b_3$, $\hat{y} = \sigma(z_3)$
- Loss: $L = (\hat{y} - y_{true})^2$

In [ ]:
# --- 2b: Forward pass ---
def np_sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def np_sigmoid_deriv(x):
    s = np_sigmoid(x)
    return s * (1.0 - s)

# Fixed parameters
x_val = 1.5
y_true = 0.8
w1, b1 = 0.5, 0.1
w2, b2 = -0.3, 0.2
w3, b3 = 0.7, -0.1

# Forward pass
z1 = w1 * x_val + b1
a1 = np_sigmoid(z1)
z2 = w2 * a1 + b2
a2 = np_sigmoid(z2)
z3 = w3 * a2 + b3
y_pred = np_sigmoid(z3)
L = (y_pred - y_true) ** 2

print('=== FORWARD PASS ===')
print(f'z1 = {z1:.6f}, a1 = {a1:.6f}')
print(f'z2 = {z2:.6f}, a2 = {a2:.6f}')
print(f'z3 = {z3:.6f}, y_pred = {y_pred:.6f}')
print(f'L = {L:.6f}')

In [ ]:
# --- 2b: Backward pass ---
# dL/dy_pred = 2 * (y_pred - y_true)
dL_dy_pred = 2 * (y_pred - y_true)

# dL/dz3 = dL/dy_pred * sigmoid'(z3)
dL_dz3 = dL_dy_pred * np_sigmoid_deriv(z3)

# dL/dw3 = dL/dz3 * a2
dL_dw3 = dL_dz3 * a2

# dL/db3 = dL/dz3
dL_db3 = dL_dz3

# dL/da2 = dL/dz3 * w3
dL_da2 = dL_dz3 * w3

# YOUR CODE HERE: Continue computing all remaining gradients
# dL/dz2 = dL/da2 * sigmoid'(z2)
# dL/dw2 = dL/dz2 * a1
# dL/db2 = dL/dz2
# dL/da1 = dL/dz2 * w2
# dL/dz1 = dL/da1 * sigmoid'(z1)
# dL/dw1 = dL/dz1 * x
# dL/db1 = dL/dz1
# dL/dx = dL/dz1 * w1

print('\n=== BACKWARD PASS ===')
print(f'dL/dw3 = {dL_dw3:.6f}')
print(f'dL/db3 = {dL_db3:.6f}')
# Print all other gradients...

In [ ]:
# --- 2b: PyTorch verification ---
x_t = torch.tensor(1.5, dtype=torch.float64)
w1_t = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(0.1, dtype=torch.float64, requires_grad=True)
w2_t = torch.tensor(-0.3, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(0.2, dtype=torch.float64, requires_grad=True)
w3_t = torch.tensor(0.7, dtype=torch.float64, requires_grad=True)
b3_t = torch.tensor(-0.1, dtype=torch.float64, requires_grad=True)

z1_t = w1_t * x_t + b1_t
a1_t = torch.sigmoid(z1_t)
z2_t = w2_t * a1_t + b2_t
a2_t = torch.sigmoid(z2_t)
z3_t = w3_t * a2_t + b3_t
y_pred_t = torch.sigmoid(z3_t)
L_t = (y_pred_t - 0.8) ** 2

L_t.backward()

print('\n=== PYTORCH VERIFICATION ===')
print(f'dL/dw3: manual = {dL_dw3:.6f}, pytorch = {w3_t.grad.item():.6f}')
print(f'dL/db3: manual = {dL_db3:.6f}, pytorch = {b3_t.grad.item():.6f}')
# Compare all remaining gradients...

**Vanishing/Exploding Gradients:** If $w_3$ is very small, what happens to gradients in earlier layers? If very large?

*YOUR ANSWER HERE (4-6 sentences)*

---
## Exercise 3: Gradient Descent on 2D Functions

In [ ]:
def gradient_descent(f, grad_f, x0, lr=0.01, n_steps=100):
    """Vanilla gradient descent.

    Returns:
        trajectory: array of points visited, shape (n_steps+1, dim)
        losses: array of function values, shape (n_steps+1,)
    """
    trajectory = [x0.copy()]
    losses = [f(x0)]
    x = x0.copy()
    for _ in range(n_steps):
        g = grad_f(x)
        x = x - lr * g
        trajectory.append(x.copy())
        losses.append(f(x))
    return np.array(trajectory), np.array(losses)


# --- Test functions ---
def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(xy):
    x, y = xy
    return np.array([-2*(1 - x) - 400*x*(y - x**2), 200*(y - x**2)])

def rastrigin(xy):
    x, y = xy
    return 20 + (x**2 - 10*np.cos(2*np.pi*x)) + (y**2 - 10*np.cos(2*np.pi*y))

def rastrigin_grad(xy):
    x, y = xy
    return np.array([2*x + 20*np.pi*np.sin(2*np.pi*x),
                     2*y + 20*np.pi*np.sin(2*np.pi*y)])

def ill_conditioned_quadratic(xy):
    x, y = xy
    return 0.5 * x**2 + 10 * y**2

def ill_conditioned_grad(xy):
    x, y = xy
    return np.array([x, 20*y])

In [ ]:
def plot_trajectory(f, trajectory, title, xlim, ylim, levels=50,
                    true_min=None):
    """Plot contour lines with optimization trajectory overlaid."""
    fig, ax = plt.subplots(figsize=(8, 6))
    x_range = np.linspace(xlim[0], xlim[1], 200)
    y_range = np.linspace(ylim[0], ylim[1], 200)
    X, Y = np.meshgrid(x_range, y_range)
    Z = np.array([[f(np.array([xi, yi])) for xi in x_range] for yi in y_range])

    ax.contourf(X, Y, Z, levels=levels, cmap='viridis')
    ax.plot(trajectory[:, 0], trajectory[:, 1], 'r.-', markersize=3, linewidth=0.8)
    ax.plot(trajectory[0, 0], trajectory[0, 1], 'g*', markersize=15, label='Start')
    ax.plot(trajectory[-1, 0], trajectory[-1, 1], 'r*', markersize=15, label='End')
    if true_min is not None:
        ax.plot(true_min[0], true_min[1], 'kX', markersize=12, label='Minimum')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()
    plt.tight_layout()
    return fig

In [ ]:
# --- Exercise 3: Run GD on each function ---
# YOUR CODE HERE
# 1. Rosenbrock: lr=0.001, start=(-1,-1), 5000 steps
# 2. Rastrigin: lr=0.01, start=(3,3), 200 steps
# 3. Ill-conditioned: lr=0.01, start=(5,1), 200 steps
# Plot trajectory + loss vs step for each
pass

**Why does GD oscillate on the ill-conditioned quadratic? What role does the Hessian's condition number play?**

*YOUR ANSWER HERE (4-6 sentences)*

---
## Exercise 4: Optimizer Showdown

### 4a: Implement Four Optimizers

In [ ]:
class VanillaGD:
    def __init__(self, lr=0.01):
        self.lr = lr

    def step(self, params, grads):
        return params - self.lr * grads


class Momentum:
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = None

    def step(self, params, grads):
        if self.v is None:
            self.v = np.zeros_like(params)
        self.v = self.beta * self.v + grads
        return params - self.lr * self.v


class RMSProp:
    def __init__(self, lr=0.01, beta=0.99, epsilon=1e-8):
        self.lr = lr
        self.beta = beta
        self.epsilon = epsilon
        self.s = None

    def step(self, params, grads):
        """s = beta*s + (1-beta)*grads^2; params -= lr * grads / (sqrt(s) + eps)"""
        if self.s is None:
            self.s = np.zeros_like(params)
        # YOUR CODE HERE
        pass


class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        """Adam = Momentum + RMSProp + bias correction."""
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        self.t += 1
        # YOUR CODE HERE
        # 1. m = beta1*m + (1-beta1)*grads
        # 2. v = beta2*v + (1-beta2)*grads^2
        # 3. m_hat = m / (1 - beta1^t)
        # 4. v_hat = v / (1 - beta2^t)
        # 5. return params - lr * m_hat / (sqrt(v_hat) + eps)
        pass

In [ ]:
# --- 4b: Race on ill-conditioned quadratic ---
optimizers = {
    'Vanilla GD': VanillaGD(lr=0.05),
    'Momentum': Momentum(lr=0.05, beta=0.9),
    'RMSProp': RMSProp(lr=0.1, beta=0.99),
    'Adam': Adam(lr=0.1, beta1=0.9, beta2=0.999)
}

results = {}
x0 = np.array([5.0, 1.0])

for name, opt in optimizers.items():
    x = x0.copy()
    trajectory = [x.copy()]
    losses = [ill_conditioned_quadratic(x)]
    for _ in range(200):
        g = ill_conditioned_grad(x)
        x = opt.step(x, g)
        trajectory.append(x.copy())
        losses.append(ill_conditioned_quadratic(x))
    results[name] = {'trajectory': np.array(trajectory), 'losses': np.array(losses)}

# YOUR CODE HERE: Plot trajectory comparison and convergence comparison
pass

In [ ]:
# --- 4c: Race on Rosenbrock ---
# YOUR CODE HERE
# Run all 4 optimizers on Rosenbrock, start=(-1,-1), 5000 steps
# Tune learning rates for each
pass

**Which optimizer performed best on each function? Why?**

*YOUR ANSWER HERE (6-8 sentences)*

In [ ]:
# --- 4d: Adam ablation study ---
# YOUR CODE HERE
# 1. Full Adam
# 2. No bias correction
# 3. No second moment (v_hat=1) -> like Momentum
# 4. No first moment (m_hat=grads) -> like RMSProp
# Run on ill-conditioned quadratic, plot convergence
pass

---
## Exercise 5: Convexity and Local Minima

In [ ]:
# --- 5a: Classify functions as convex/non-convex ---
# YOUR CODE HERE
# Plot f(x) = x^2, |x|, x^4 - 2x^2, x^3, log(1+exp(x))
# For each, verify convexity condition with two test points
pass

In [ ]:
# --- 5b: Multiple local minima ---
def f_multi(x):
    return x**4 - 2*x**2

def f_multi_prime(x):
    return 4*x**3 - 4*x

# Critical points: x = 0, 1, -1
# f''(x) = 12x^2 - 4
for x_crit in [-1, 0, 1]:
    f_pp = 12*x_crit**2 - 4
    kind = 'minimum' if f_pp > 0 else 'maximum'
    print(f"x = {x_crit}: f''(x) = {f_pp}, {kind}")

# GD from different starting points
starts = [-2.0, -0.5, 0.01, 0.5, 2.0]
for x0 in starts:
    x = x0
    for _ in range(1000):
        x = x - 0.01 * f_multi_prime(x)
    print(f'Start: {x0:+.1f} -> converged to x = {x:.4f}, f(x) = {f_multi(x):.4f}')

**Why does gradient descent still work for non-convex neural networks?**

*YOUR ANSWER HERE (4-6 sentences)*

---
## Exercise 6: Saddle Points and SGD Noise

In [ ]:
# --- 6a: Visualize saddle point f(x,y) = x^2 - y^2 ---
fig = plt.figure(figsize=(16, 6))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)
Z = X**2 - Y**2
ax1.plot_surface(X, Y, Z, cmap='coolwarm', alpha=0.8)
ax1.set_title('$f(x,y) = x^2 - y^2$ (Saddle Point)')

# Contour + gradient field
ax2 = fig.add_subplot(122)
ax2.contour(X, Y, Z, levels=20, cmap='coolwarm')
step = 5
Xq, Yq = X[::step, ::step], Y[::step, ::step]
U = -2 * Xq
V = 2 * Yq
ax2.quiver(Xq, Yq, U, V, alpha=0.5)
ax2.set_title('Contour + Negative Gradient Field')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.tight_layout()
plt.show()

In [ ]:
# --- 6b: GD vs SGD near saddle ---
def run_optimization(start, lr, n_steps, noise_std=0.0, seed=42):
    np.random.seed(seed)
    x, y = start
    trajectory = [(x, y)]
    for _ in range(n_steps):
        grad_x = 2 * x + np.random.normal(0, noise_std)
        grad_y = -2 * y + np.random.normal(0, noise_std)
        x -= lr * grad_x
        y -= lr * grad_y
        trajectory.append((x, y))
    return np.array(trajectory)

# YOUR CODE HERE
# 1. Pure GD: noise_std=0 (gets stuck)
# 2. SGD: noise_std=0.1 (escapes)
# Plot both trajectories on contour
pass

In [ ]:
# --- 6c: Effect of noise magnitude ---
# YOUR CODE HERE
# Test noise_std = [0, 0.01, 0.05, 0.1, 0.5, 1.0]
# For each, run 100 times, measure escape time
# Plot noise level vs average escape time
pass

**How does noise relate to batch size in SGD? What does this suggest for batch size selection?**

*YOUR ANSWER HERE (4-6 sentences)*

In [ ]:
# --- 6d: High-dimensional saddle point ---
# YOUR CODE HERE
# f(x) = sum(x[:5]^2) - sum(x[5:]^2), 10 dimensions
# Compare GD vs SGD escape from near origin
pass

**In high-dimensional space, what fraction of Hessian eigenvalues must be negative for a saddle point?**

*YOUR ANSWER HERE (3-5 sentences)*